In [ ]:
import math
from typing import Optional

import torch
import torch.nn as nn

class PhyloAttention(nn.Module):
    """
    Multi-head attention with:
    - ALiBi linear bias (no learned positional embeddings)
    - Phylogenetic distance bias (learned scalar weight)
    - Causal masking
    """
    def __init__(self, embed_dim: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        assert self.head_dim * num_heads == embed_dim, "embed_dim must be divisible by num_heads"

        self.scale = self.head_dim ** -0.5

        self.qkv_proj = nn.Linear(embed_dim, embed_dim * 3)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

        # Learned strength of phylogenetic bias
        self.phylo_alpha = nn.Parameter(torch.tensor(0.5))

    def forward(
        self,
        x: torch.Tensor,                    # (B, L, D)
        phylo_dists: torch.Tensor | None,   # (B, num_reps) or (B, L, L), or None
        alibi_bias: torch.Tensor,           # (num_heads, L, L)
        attn_mask: Optional[torch.Tensor] = None,  # (1, 1, L, L) causal
    ):
        B, L, D = x.shape

        qkv = self.qkv_proj(x).reshape(B, L, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)          # (3, B, heads, L, head_dim)
        q, k, v = qkv.unbind(0)

        # Scale attention temperature by phylogenetic distance; clamp keeps temp positive.
        if phylo_dists is not None:
            if phylo_dists.dim() == 2:             # (B, num_reps)
                phylo_scalar = phylo_dists.mean(dim=-1).view(B, 1, 1, 1)
            else:                                  # (B, L, L)
                phylo_scalar = phylo_dists.mean(dim=(-1, -2)).view(B, 1, 1, 1)
            temp = (1.0 + self.phylo_alpha * phylo_scalar).clamp(min=1e-6)
        else:
            temp = 1.0

        scores = (q @ k.transpose(-2, -1)) * (self.scale * temp)  # (B, heads, L, L)
        scores = scores + alibi_bias.unsqueeze(0)                  # ALiBi positional bias

        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask == 0, float('-inf'))

        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        out = attn @ v                             # (B, heads, L, head_dim)
        out = out.transpose(1, 2).reshape(B, L, D)
        return self.out_proj(out)


In [4]:
import torch.nn as nn

class PhyloGenBlock(nn.Module):
    def __init__(
        self,
        embed_dim: int,
        num_heads: int,
        ff_dim: int = None,
        dropout: float = 0.1,
    ):
        super().__init__()
        ff_dim = ff_dim or embed_dim * 4

        self.attn = PhyloAttention(embed_dim, num_heads, dropout=dropout)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, embed_dim),
            nn.Dropout(dropout),
        )

        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x, phylo_dists, alibi_bias, attn_mask):
        # Attention path
        attn_out = self.attn(self.norm1(x), phylo_dists, alibi_bias, attn_mask)
        x = x + attn_out

        # Feed-forward path
        ffn_out = self.ffn(self.norm2(x))
        x = x + ffn_out

        return x


In [5]:
from typing import Dict, List, Optional, Tuple
import json
import torch

class ProteinTokenizer:
    def __init__(
        self,
        vocab: Optional[Dict[str, int]] = None,
        special_tokens: Optional[List[str]] = None,
        pad_token: str = "[PAD]",
        unk_token: str = "[UNK]",
        bos_token: str = "[BOS]",
        eos_token: str = "[EOS]",
    ):
        if vocab is None:
            vocab = self._build_default_vocab(pad_token, unk_token, bos_token, eos_token, special_tokens or [])

        self.vocab = vocab
        self.inverse_vocab = {v: k for k, v in vocab.items()}

        self.pad_token = pad_token
        self.unk_token = unk_token
        self.bos_token = bos_token
        self.eos_token = eos_token

        self.pad_token_id = vocab[pad_token]
        self.unk_token_id = vocab[unk_token]
        self.bos_token_id = vocab[bos_token]
        self.eos_token_id = vocab[eos_token]

    def _build_default_vocab(self, pad_token: str, unk_token: str, bos_token: str, eos_token: str, extra_specials: List[str]) -> Dict[str, int]:
        # Standard amino acids + stop
        standard_aas = list("ACDEFGHIKLMNPQRSTVWY*")
        # Common ambiguous / special
        ambiguous = list("XBJZOU")

        base = standard_aas + ambiguous

        # Protein boundary markers
        boundaries = ["<PROT>", "</PROT>"]

        # Core special tokens
        specials = [
            bos_token, eos_token, pad_token, unk_token,
            "[MUT]", "[RESISTANT]", "[SUSCEPTIBLE]",
            "[CIPRO]", "[FLUOROQUINOLONE]", "[AMPC]", "[MEROPENEM]",  # antibiotics
            "[SPECIES_ECOLI]", "[PHYLO_REF]", "[SEP]"
        ]

        # Add user-requested extras
        specials += extra_specials

        all_tokens = base + boundaries + specials

        vocab = {token: i for i, token in enumerate(all_tokens)}
        return vocab

    @property
    def vocab_size(self) -> int:
        return len(self.vocab)

    def encode(
        self,
        sequence: str,
        add_special_tokens: bool = True,
        conditioning: Optional[List[str]] = None,
    ) -> torch.LongTensor:
        """
        Encode a proteome string (with <PROT>...</PROT> markers)

        Args:
            sequence: concatenated proteome string
            add_special_tokens: whether to add [BOS]/[EOS]
            conditioning: list of conditioning tokens e.g. ["[SPECIES_ECOLI]", "[CIPRO]", "[RESISTANT]"]

        Returns:
            torch.LongTensor of token ids
        """
        tokens = []

        if add_special_tokens:
            tokens.append(self.bos_token_id)

        if conditioning:
            for cond in conditioning:
                if cond in self.vocab:
                    tokens.append(self.vocab[cond])
                else:
                    tokens.append(self.unk_token_id)

        # Character-level tokenization, but recognize full special tokens like <PROT>
        i = 0
        while i < len(sequence):
            # Check for known multi-char tokens first (greedy longest match)
            matched = False
            for special in ["<PROT>", "</PROT>"] + [t for t in self.vocab if t.startswith("[")]:
                if sequence[i:].startswith(special):
                    tokens.append(self.vocab[special])
                    i += len(special)
                    matched = True
                    break

            if not matched:
                aa = sequence[i].upper()
                tokens.append(self.vocab.get(aa, self.unk_token_id))
                i += 1

        if add_special_tokens:
            tokens.append(self.eos_token_id)

        return torch.tensor(tokens, dtype=torch.long)
    
    def encode_fast(
        self,
        sequence: str,
        add_special_tokens: bool = True,
        conditioning: Optional[List[str]] = None,
    ) -> torch.LongTensor:
        tokens = []

        if add_special_tokens:
            tokens.append(self.bos_token_id)

        if conditioning:
            for cond in conditioning:
                tokens.append(self.vocab.get(cond, self.unk_token_id))

        # Split on protein boundaries to avoid slow per-char checks
        parts = sequence.split("<PROT>")
        for part in parts:
            if not part.strip():
                continue
            subparts = part.split("</PROT>")
            for i, sub in enumerate(subparts):
                if i < len(subparts) - 1:  # before each </PROT>
                    # Add <PROT> only at start of real protein
                    if sub.strip():
                        tokens.append(self.vocab["<PROT>"])
                    # Tokenize AAs fast
                    for aa in sub.upper():
                        if aa in self.vocab:
                            tokens.append(self.vocab[aa])
                        else:
                            tokens.append(self.unk_token_id)
                    tokens.append(self.vocab["</PROT>"])
                else:
                    # Leftover after last </PROT> — rare, but handle
                    for aa in sub.upper():
                        tokens.append(self.vocab.get(aa, self.unk_token_id))

        if add_special_tokens:
            tokens.append(self.eos_token_id)

        return torch.tensor(tokens, dtype=torch.long)

    def decode(
        self,
        token_ids: torch.Tensor | List[int],
        skip_special: bool = False,
        remove_boundaries: bool = False,
    ) -> str:
        if isinstance(token_ids, torch.Tensor):
            token_ids = token_ids.tolist()

        tokens = []
        for tid in token_ids:
            token = self.inverse_vocab.get(tid, self.unk_token)
            if skip_special and token in {self.bos_token, self.eos_token, self.pad_token}:
                continue
            tokens.append(token)

        text = "".join(tokens)

        if remove_boundaries:
            text = text.replace("<PROT>", "").replace("</PROT>", "")

        return text

    def save(self, path: str):
        config = {
            "vocab": self.vocab,
            "pad_token": self.pad_token,
            "unk_token": self.unk_token,
            "bos_token": self.bos_token,
            "eos_token": self.eos_token,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(config, f, indent=2)

    @classmethod
    def load(cls, path: str):
        with open(path, "r", encoding="utf-8") as f:
            config = json.load(f)
        return cls(
            vocab=config["vocab"],
            pad_token=config["pad_token"],
            unk_token=config["unk_token"],
            bos_token=config["bos_token"],
            eos_token=config["eos_token"],
        )


# ────────────────────────────────────────────────
# Quick test
# ────────────────────────────────────────────────

if __name__ == "__main__":
    tokenizer = ProteinTokenizer()

    print("Vocab size:", tokenizer.vocab_size)
    print("Some important ids:")
    print("  [BOS]:", tokenizer.bos_token_id)
    print("  [CIPRO]:", tokenizer.vocab.get("[CIPRO]"))
    print("  <PROT>:", tokenizer.vocab.get("<PROT>"))
    print("  A:", tokenizer.vocab.get("A"))

    # Example proteome snippet
    example = "<PROT>MATKTTTVNG</PROT><PROT>MFVKLLRSVA</PROT>"

    conditioning = ["[SPECIES_ECOLI]", "[CIPRO]", "[RESISTANT]"]

    encoded = tokenizer.encode(example, conditioning=conditioning)
    print("\nEncoded:", encoded.tolist())

    decoded = tokenizer.decode(encoded, skip_special=True)
    print("Decoded:", decoded)

    tokenizer.save("tokenizer.json")


Vocab size: 43
Some important ids:
  [BOS]: 29
  [CIPRO]: 36
  <PROT>: 27
  A: 0

Encoded: [29, 40, 36, 34, 27, 10, 0, 16, 8, 16, 16, 16, 17, 11, 5, 28, 27, 10, 4, 17, 8, 9, 9, 14, 15, 17, 0, 28, 30]
Decoded: [SPECIES_ECOLI][CIPRO][RESISTANT]<PROT>MATKTTTVNG</PROT><PROT>MFVKLLRSVA</PROT>


In [6]:
import sys
import pickle
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

class ProteomeDataset(Dataset):
    def __init__(
        self,
        csv_path: str,
        tokenizer: 'ProteinTokenizer',
        chunk_size: int = 1024,
        overlap: int = 256,
        phylo_pkl: str = None,
        mode: str = "finetune",
        max_samples: int = None,
        start_idx: int = 0,
        use_mutated_only: bool = False,
        cache_dir: str = None,
        force_recompute: bool = False,
    ):
        if phylo_pkl is None:
            phylo_pkl = str(Path(__file__).parent.parent / "data" / "gtdb_data" / "ecoli_phylo_distances.pkl")

        df = pd.read_csv(csv_path, dtype={'genome_id': str})

        if mode == "finetune" and use_mutated_only:
            if 'reversions_applied' in df.columns:
                df = df[df['reversions_applied'] == True].reset_index(drop=True)
                print(f"Filtered to {len(df)} genomes with reversions applied (mutated pairs)")
            else:
                print("Warning: 'reversions_applied' column not found — using all rows")

        if max_samples is not None:
            df = df.iloc[start_idx:start_idx + max_samples].reset_index(drop=True)
            print(f"Loading chunk: samples {start_idx} to {start_idx + len(df)}")
        else:
            df = df.iloc[start_idx:].reset_index(drop=True)

        self.df = df
        self.tokenizer = tokenizer
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.mode = mode
        self.stride = chunk_size - overlap

        self.phylo_matrix = pickle.load(open(phylo_pkl, "rb"))
        self.num_reps = self.phylo_matrix.shape[0]

        self.genome_to_phylo_idx = {}
        for idx in range(len(self.df)):
            genome_id = self.df.iloc[idx]['genome_id']
            h = int(hashlib.sha256(genome_id.encode()).hexdigest(), 16)
            self.genome_to_phylo_idx[idx] = h % self.num_reps

        # Caching
        if cache_dir is None:
            cache_dir = str(Path(csv_path).parent)
        cache_dir = Path(cache_dir)
        cache_dir.mkdir(exist_ok=True, parents=True)

        cache_key_parts = [
            Path(csv_path).stem,
            f"mode_{mode}",
            f"chunk{chunk_size}_overlap{overlap}",
            f"mutated_only_{use_mutated_only}",
            f"start{start_idx}_max{max_samples or 'all'}",
        ]
        cache_filename = "_".join(cache_key_parts) + ".chunks.pkl"
        self.cache_path = cache_dir / cache_filename

        if self.cache_path.exists() and not force_recompute:
            try:
                print(f"Loading cached useful chunks from: {self.cache_path}")
                with open(self.cache_path, "rb") as f:
                    self.chunk_indices = pickle.load(f)
                print(f"→ Loaded {len(self.chunk_indices):,} cached chunks successfully")
                return
            except Exception as e:
                print(f"Cache load failed — recomputing")

        # Compute chunks (mutation-centered)
        self.chunk_indices = []
        print("Computing useful chunks...")
        MAX_CHUNKS_PER_GENOME = 50

        for g_idx in range(len(self.df)):
            print(f"  Processing genome {g_idx+1}/{len(self.df)} ...")
            row = self.df.iloc[g_idx]

            if mode == "finetune":
                cond = ["[SPECIES_ECOLI]", "[CIPRO]", "[RESISTANT]"]
                bos_id = tokenizer.bos_token_id
                sep_id = tokenizer.vocab["[SEP]"]

                # Pure proteome tokens (with <PROT>/</PROT>, NO BOS, NO cond, NO EOS)
                unmut_prot = tokenizer.encode_fast(row['unmutated_proteome'], add_special_tokens=False)
                mut_prot   = tokenizer.encode_fast(row['mutated_proteome'],   add_special_tokens=False)

                mutation_token_pos = [
                    t for t in range(min(len(unmut_prot), len(mut_prot)))
                    if unmut_prot[t] != mut_prot[t]
                ]
                num_mutations = len(mutation_token_pos)
                print(f"    → Encoded | mutations found: {num_mutations}")

                if num_mutations == 0:
                    continue

                cond_ids = torch.tensor([tokenizer.vocab[c] for c in cond], dtype=torch.long)
                # Prefix up to and including SEP
                prefix = torch.cat([
                    torch.tensor([bos_id], dtype=torch.long),
                    cond_ids,
                    unmut_prot,
                    torch.tensor([sep_id], dtype=torch.long)
                ])
                sep_token_pos = len(prefix) - 1

                token_len = len(prefix) + len(mut_prot)   # full length for chunk bounds

                global_mutation_pos = [sep_token_pos + 1 + t for t in mutation_token_pos]

                print(f"    → Generating chunks around {num_mutations} mutations...")

                genome_added = 0
                for mut_pos in global_mutation_pos:
                    for offset in range(-chunk_size//2, chunk_size//2 + 1, self.stride//2):
                        start = max(0, mut_pos - chunk_size//2 + offset)
                        end = start + chunk_size
                        if end > token_len: continue
                        chunk_muts = [p for p in global_mutation_pos if start <= p < end]
                        if len(chunk_muts) == 0: continue
                        if end <= sep_token_pos + 1: continue  # must overlap continuation

                        self.chunk_indices.append((g_idx, start, sep_token_pos))
                        genome_added += 1
                        if genome_added % 5 == 0 or genome_added == 1:
                            print(f"      Added chunk {genome_added} at start={start} (covers {len(chunk_muts)} muts)")
                        if genome_added >= MAX_CHUNKS_PER_GENOME:
                            break
                    if genome_added >= MAX_CHUNKS_PER_GENOME: break

                print(f"    → Genome {g_idx+1} done | added {genome_added} chunks")

        print(f"Saving chunk cache to: {self.cache_path}")
        with open(self.cache_path, "wb") as f:
            pickle.dump(self.chunk_indices, f)
        print(f"→ Saved {len(self.chunk_indices):,} useful chunks")

        print(f"Dataset ready (finetune mode): {len(self.chunk_indices):,} useful chunks from {len(df)} genomes")

    def __getitem__(self, idx):
        genome_idx, chunk_start, global_sep_pos = self.chunk_indices[idx]
        row = self.df.iloc[genome_idx]

        cond = ["[SPECIES_ECOLI]", "[CIPRO]", "[RESISTANT]"]
        bos_id = self.tokenizer.bos_token_id
        sep_id = self.tokenizer.vocab["[SEP]"]

        cond_ids = torch.tensor([self.tokenizer.vocab[c] for c in cond], dtype=torch.long)

        # Pure proteome (continuation)
        unmut_prot = self.tokenizer.encode_fast(row['unmutated_proteome'], add_special_tokens=False)
        mut_prot   = self.tokenizer.encode_fast(row['mutated_proteome'],   add_special_tokens=False)

        sep_tensor = torch.tensor([sep_id], dtype=torch.long)
        bos_tensor = torch.tensor([bos_id], dtype=torch.long)

        # INPUT:  ... unmutated continuation after SEP
        # LABELS: ... mutated   continuation after SEP
        input_full = torch.cat([bos_tensor, cond_ids, unmut_prot, sep_tensor, unmut_prot])
        label_full = torch.cat([bos_tensor, cond_ids, unmut_prot, sep_tensor, mut_prot])

        chunk_end = chunk_start + self.chunk_size
        input_chunk = input_full[chunk_start:chunk_end].clone()
        labels      = label_full[chunk_start:chunk_end].clone()

        rel_sep = global_sep_pos - chunk_start if chunk_start <= global_sep_pos < chunk_end else -1

        pad_len = self.chunk_size - len(input_chunk)
        if pad_len > 0:
            pad = torch.full((pad_len,), self.tokenizer.pad_token_id, dtype=torch.long)
            input_chunk = torch.cat([input_chunk, pad])
            labels      = torch.cat([labels, pad])

        phylo_idx = self.genome_to_phylo_idx[genome_idx]
        phylo_dist = torch.tensor(self.phylo_matrix[phylo_idx], dtype=torch.float)

        return {
            'input_ids': input_chunk,
            'labels': labels,
            'phylo_dist': phylo_dist,
            'genome_id': row['genome_id'],
            'sep_pos': torch.tensor([rel_sep], dtype=torch.long),
            'chunk_start_global': torch.tensor([chunk_start], dtype=torch.long),
        }

    def __len__(self):
        return len(self.chunk_indices)

def collate_fn(batch):
    return {
        'input_ids': torch.stack([b['input_ids'] for b in batch]),
        'labels': torch.stack([b['labels'] for b in batch]),
        'phylo_dist': torch.stack([b['phylo_dist'] for b in batch]),
        'genome_id': [b['genome_id'] for b in batch],
        'sep_pos': torch.tensor([b['sep_pos'].item() for b in batch]),
    }

In [7]:
"""
ALiBi (Attention with Linear Biases) Embedder for DNA Sequences

Implements ALiBi as described in "Train Short, Test Long: Attention with Linear Biases
Enables Input Length Extrapolation" (Press et al., 2022)
https://arxiv.org/abs/2108.12409

ALiBi does not use positional embeddings. Instead, it adds a linear bias
to attention scores based on distance between positions.
"""

import math

import torch
import torch.nn as nn


class ALiBiEmbedder(nn.Module):
    def __init__(
        self,
        vocab_size: int = 9,
        embed_dim: int = 256,
        max_len: int = 1024,
        dropout: float = 0.1,
        num_heads: int = 8,
    ):
        """
        DNA Embedder with ALiBi (no positional encodings).

        Args:
            vocab_size: Number of tokens in vocabulary
            embed_dim: Dimension of embedding
            max_len: Maximum sequence length (for pre-computing biases)
            dropout: Dropout probability
            num_heads: Number of attention heads (for computing slopes)
        """
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.max_len = max_len

        # Token embedding only (no positional encoding)
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.dropout = nn.Dropout(dropout)

        # Compute ALiBi slopes for each attention head
        # Slopes are geometric sequence: m_i = 2^(-8i/n) where n is num_heads
        slopes = self._get_alibi_slopes(num_heads)
        self.register_buffer("slopes", slopes)

        # Precompute bias matrix for efficiency
        # Shape: (num_heads, max_len, max_len)
        bias_matrix = self._build_alibi_bias_matrix(max_len, slopes)
        self.register_buffer("bias_matrix", bias_matrix)

    def _get_alibi_slopes(self, num_heads: int) -> torch.Tensor:
        """
        Compute ALiBi slopes for attention heads.

        For num_heads = 8:
        slopes = [2^-1, 2^-2, 2^-3, 2^-4, 2^-5, 2^-6, 2^-7, 2^-8]
               = [0.5, 0.25, 0.125, 0.0625, ...]

        Args:
            num_heads: Number of attention heads

        Returns:
            Tensor of slopes, shape (num_heads,)
        """

        # Compute the base slope factor
        # For 8 heads: 2^(-8/8) = 2^-1 = 0.5
        def get_slopes_power_of_2(n):
            start = 2 ** (-(2 ** -(math.log2(n) - 3)))
            ratio = start
            return [start * (ratio**i) for i in range(n)]

        # For non-power-of-2 heads, use closest power of 2 and interpolate
        if math.log2(num_heads).is_integer():
            slopes = get_slopes_power_of_2(num_heads)
        else:
            # Use closest power of 2
            closest_power_of_2 = 2 ** math.floor(math.log2(num_heads))
            slopes = get_slopes_power_of_2(closest_power_of_2)

            # Add extra slopes by interpolation
            extra_slopes_count = num_heads - closest_power_of_2
            extra_base = 2 ** (-(2 ** -(math.log2(2 * closest_power_of_2) - 3)))
            extra_slopes = [
                extra_base * (extra_base**i) for i in range(extra_slopes_count)
            ]
            slopes = slopes + extra_slopes

        return torch.tensor(slopes, dtype=torch.float32)

    def _build_alibi_bias_matrix(
        self, max_len: int, slopes: torch.Tensor
    ) -> torch.Tensor:
        """
        Build the ALiBi bias matrix for all heads.

        The bias for position i attending to position j is:
        bias[i, j] = -slope * |i - j|

        Args:
            max_len: Maximum sequence length
            slopes: Slopes for each head, shape (num_heads,)

        Returns:
            Bias matrix of shape (num_heads, max_len, max_len)
        """
        # Create position distance matrix
        # distances[i, j] = |i - j|
        positions = torch.arange(max_len).unsqueeze(1)  # (max_len, 1)
        distances = torch.abs(positions - positions.T)  # (max_len, max_len)

        # Apply slopes to get biases for each head
        # Shape: (num_heads, max_len, max_len)
        biases = -slopes.unsqueeze(1).unsqueeze(2) * distances.unsqueeze(0)

        return biases

    def get_alibi_bias(self, seq_len: int) -> torch.Tensor:
        """
        Get ALiBi bias matrix for current sequence length.

        This can be added to attention scores in the transformer.

        Args:
            seq_len: Current sequence length

        Returns:
            Bias matrix of shape (num_heads, seq_len, seq_len)
        """
        if seq_len <= self.max_len:
            # Use precomputed bias
            return self.bias_matrix[:, :seq_len, :seq_len]
        else:
            # Recompute for longer sequences (extrapolation)
            slopes = self.slopes
            return self._build_alibi_bias_matrix(seq_len, slopes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor (batch, seq_len)
        Returns:
            Tensor (batch, seq_len, embed_dim) WITHOUT positional encodings.
            ALiBi biases are added in the attention layer, not here.
        """
        batch_size, seq_len = x.size()

        # Get token embeddings and scale
        # Note: No positional encoding added here!
        embedded = self.embedding(x) * math.sqrt(self.embed_dim)

        # Apply dropout
        embedded = self.dropout(embedded)

        return embedded


class ALiBiEmbedderSimple(nn.Module):
    """
    Simplified ALiBi embedder that just returns token embeddings.
    The bias computation is deferred to the attention mechanism.
    """

    def __init__(
        self,
        vocab_size: int = 9,
        embed_dim: int = 256,
        dropout: float = 0.1,
    ):
        """
        Simple DNA Embedder for ALiBi (no positional information).

        Args:
            vocab_size: Number of tokens in vocabulary
            embed_dim: Dimension of embedding
            dropout: Dropout probability
        """
        super().__init__()
        self.embed_dim = embed_dim

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor (batch, seq_len)
        Returns:
            Tensor (batch, seq_len, embed_dim) with only token embeddings
        """
        # Token embeddings with scaling
        embedded = self.embedding(x) * math.sqrt(self.embed_dim)

        # Apply dropout
        embedded = self.dropout(embedded)

        return embedded


In [8]:
import math
from typing import Optional
from pathlib import Path

import torch
import torch.nn as nn

class PhyloGen(nn.Module):
    """
    Decoder-only transformer with:
    - Token embedding
    - ALiBi positional bias
    - Phylogenetic attention bias
    - Standard transformer blocks
    """
    def __init__(
        self,
        vocab_size: int,
        tokenizer=None, # pass tokenizer for dynamic pad id
        embed_dim: int = 256,
        num_heads: int = 8,
        num_layers: int = 6,
        max_seq_len: int = 2048,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.embed_dim = embed_dim
        self.num_heads = num_heads

        if tokenizer is not None:
            self.pad_token_id = tokenizer.pad_token_id
        else:
            self.pad_token_id = 31  # fallback

        # Token embedding
        self.token_embed = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=self.pad_token_id
        )
        self.embed_dropout = nn.Dropout(dropout)

        # ALiBi component
        self.alibi = ALiBiEmbedder(
            vocab_size=vocab_size,
            embed_dim=embed_dim,
            max_len=max_seq_len,
            num_heads=num_heads,
            dropout=dropout,
        )

        # Stack of transformer blocks (these are the decoder layers)
        self.blocks = nn.ModuleList([
            PhyloGenBlock(
                embed_dim=embed_dim,
                num_heads=num_heads,
                dropout=dropout,
            )
            for _ in range(num_layers)
        ])

        # Final norm + LM head
        self.final_norm = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size, bias=False)

        # Weight tying
        self.lm_head.weight = self.token_embed.weight

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.token_embed.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.lm_head.weight, mean=0.0, std=0.02 / math.sqrt(self.embed_dim))

    def forward(
        self,
        input_ids: torch.Tensor,
        phylo_dists: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
        sep_pos: Optional[torch.Tensor] = None,  # new: (B,)
        return_dict: bool = True,
    ):
        B, L = input_ids.shape

        x = self.token_embed(input_ids) * math.sqrt(self.embed_dim)
        x = self.embed_dropout(x)

        alibi_bias = self.alibi.get_alibi_bias(L)
        attn_mask = torch.tril(torch.ones(L, L, device=x.device, dtype=torch.bool)).view(1, 1, L, L)

        for block in self.blocks:
            x = block(x, phylo_dists, alibi_bias, attn_mask)

        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()

            # Mask prefix up to separator (finetune mode)
            if sep_pos is not None:
                ignore_mask = torch.zeros_like(shift_labels, dtype=torch.bool)
                for b in range(B):
                    pos = sep_pos[b].item()
                    if pos >= 0:
                        ignore_mask[b, :pos] = True  # ignore before SEP
                shift_labels = shift_labels.masked_fill(ignore_mask, -100)

            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1)
            )

        if return_dict:
            return {"logits": logits, "loss": loss}
        return logits

    def to(self, *args, **kwargs):
        """
        Override .to() so ALiBi buffers move to the correct device too
        """
        super().to(*args, **kwargs)
        self.alibi.to(*args, **kwargs)  # important for precomputed biases
        return self

In [14]:
import sys
import torch
import json
import time
from pathlib import Path
from torch.utils.data import DataLoader
from tqdm import tqdm

sys.path.append(str(Path().resolve().parent))

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# ========================== PATHS ==========================
tokenizer_path = "tokenizer.json"
csv_path       = "ecoli_pairs_concat.csv"
phylo_pkl_path = "ecoli_phylo_distances.pkl"

tokenizer = ProteinTokenizer.load(str(tokenizer_path))
print(f"Tokenizer loaded — vocab size: {tokenizer.vocab_size}")

Using device: cuda
Tokenizer loaded — vocab size: 43


In [15]:
model = PhyloGen(
    vocab_size=tokenizer.vocab_size,
    tokenizer=tokenizer,
    embed_dim=256,
    num_heads=8,
    num_layers=6,
    max_seq_len=2048,
).to(device)

PRETRAINED_CHECKPOINT = "pretrain_step_50000.pt"

print(f"Loading pretrained checkpoint: {PRETRAINED_CHECKPOINT}")
ckpt = torch.load(PRETRAINED_CHECKPOINT, map_location=device)
pretrained_state = ckpt.get('model', ckpt)

model_state = model.state_dict()

for key in ['token_embed.weight', 'lm_head.weight', 'alibi.embedding.weight']:
    if key in pretrained_state and key in model_state:
        old = pretrained_state[key]
        new_shape = model_state[key].shape
        if old.shape != new_shape:
            print(f"Resizing {key}: {old.shape} → {new_shape}")
            new_weight = torch.zeros(new_shape, dtype=old.dtype, device=old.device)
            n = min(old.shape[0], new_shape[0])
            new_weight[:n] = old[:n]
            if new_shape[0] > old.shape[0]:
                torch.nn.init.normal_(new_weight[n:], mean=0.0, std=0.02)
            pretrained_state[key] = new_weight

model.load_state_dict(pretrained_state, strict=False)
model.lm_head.weight = model.token_embed.weight

print("✓ Pretrained checkpoint loaded and adapted")

Loading pretrained checkpoint: pretrain_step_50000.pt
Resizing token_embed.weight: torch.Size([42, 256]) → torch.Size([43, 256])
Resizing lm_head.weight: torch.Size([42, 256]) → torch.Size([43, 256])
Resizing alibi.embedding.weight: torch.Size([42, 256]) → torch.Size([43, 256])
✓ Pretrained checkpoint loaded and adapted


In [16]:
def compute_weighted_loss(
    logits,
    labels,
    input_ids,
    sep_pos,
    ignore_index=-100
):
    """Only supervises the real mutation positions after [SEP]."""
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    shift_input  = input_ids[..., 1:].contiguous()

    B, T = shift_labels.shape
    device = shift_labels.device

    # Handle sep_pos = -1 (whole chunk is continuation)
    effective_sep = torch.where(sep_pos >= 0, sep_pos, torch.zeros_like(sep_pos, dtype=torch.long))

    after_sep = (torch.arange(T, device=device).unsqueeze(0) >= effective_sep.unsqueeze(1))
    is_edit   = (shift_labels != shift_input) & (shift_labels != ignore_index)

    edit_mask = after_sep & is_edit

    if edit_mask.sum() == 0:
        return shift_logits.mean() * 0.0 + 1e-7

    loss_fct = nn.CrossEntropyLoss(ignore_index=ignore_index, reduction='none')
    loss_per_token = loss_fct(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1)
    )

    weighted = loss_per_token * edit_mask.view(-1).float()
    final_loss = weighted.sum() / edit_mask.sum().clamp(min=1.0)

    return final_loss

In [18]:
dataset = ProteomeDataset(
    csv_path=str(csv_path),
    tokenizer=tokenizer,
    chunk_size=1024,
    overlap=512,
    phylo_pkl=str(phylo_pkl_path),
    mode="finetune",
    max_samples=None,
    use_mutated_only=True,
    cache_dir="cache",
    force_recompute=False
)

print(f"Dataset ready: {len(dataset)} chunks from {len(dataset.df)} genomes")

Filtered to 382 genomes with reversions applied (mutated pairs)
Loading cached useful chunks from: cache/ecoli_pairs_concat_mode_finetune_chunk1024_overlap512_mutated_only_True_start0_maxall.chunks.pkl
→ Loaded 4,524 cached chunks successfully
Dataset ready: 4524 chunks from 382 genomes


In [22]:
loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=12,
    prefetch_factor=2,
    persistent_workers=True,
    pin_memory=True,
)

# ====================== FINAL WORKING SETUP (Strong Edit Weighting + Stability) ======================
checkpoint_dir = Path("checkpoints_finetune")
checkpoint_dir.mkdir(exist_ok=True, parents=True)

loss_log_file = Path("finetune_loss_log.json")
save_every_steps = 2000

epochs               = 3
lr                   = 5e-5
max_grad_norm        = 1.0
warmup_steps         = 500
total_steps_estimate = len(loader) * epochs * 2

if loss_log_file.exists():
    with open(loss_log_file, "r") as f:
        loss_log = json.load(f)
else:
    loss_log = []

# Freeze first 3 blocks (optional but recommended)
for i, block in enumerate(model.blocks):
    if i < 3:
        for param in block.parameters():
            param.requires_grad = False

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=lr,
    weight_decay=0.01
)

scaler = torch.amp.GradScaler(enabled=(device.type == 'cuda'))

def get_lr(step):
    if step < warmup_steps:
        return lr * float(step) / float(max(1, warmup_steps))
    progress = float(step - warmup_steps) / float(max(1, total_steps_estimate - warmup_steps))
    return lr * 0.5 * (1.0 + math.cos(math.pi * progress))

In [23]:
model.train()
global_step = 0

for epoch in range(epochs):
    epoch_num = epoch + 1
    print(f"\n=== Finetune Epoch {epoch_num} (Mask-Only-Edits Loss) ===")

    total_loss = 0.0
    steps_this_epoch = 0
    useful_batches = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch_num}", dynamic_ncols=True)

    for batch in pbar:
        global_step += 1
        steps_this_epoch += 1

        input_ids = batch['input_ids'].to(device)
        phylo     = batch['phylo_dist'].to(device)
        labels    = batch['labels'].to(device)
        sep_pos   = batch['sep_pos'].to(device)

        has_edits = (labels != input_ids).any().item()
        if not has_edits:
            continue

        useful_batches += 1

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type):
            out = model(input_ids, phylo, labels=labels, sep_pos=sep_pos)
            loss = compute_weighted_loss(out["logits"], labels, input_ids, sep_pos)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        scaler.step(optimizer)
        scaler.update()

        current_lr = get_lr(global_step)
        for param_group in optimizer.param_groups:
            param_group['lr'] = current_lr

        total_loss += loss.item()
        avg_loss = total_loss / steps_this_epoch

        # Quick edit accuracy logging
        with torch.no_grad():
            shift_logits = out["logits"][..., :-1, :]
            shift_labels = labels[..., 1:]
            shift_input  = input_ids[..., 1:]
            is_edit = (shift_labels != shift_input) & (shift_labels != -100)
            if is_edit.any():
                pred = shift_logits[is_edit].argmax(-1)
                edit_acc = (pred == shift_labels[is_edit]).float().mean().item()
                edit_count = is_edit.sum().item()
            else:
                edit_acc = 0.0
                edit_count = 0

        pbar.set_description(
            f"Epoch {epoch_num} loss:{loss.item():.4f} avg:{avg_loss:.4f} edit_acc:{edit_acc:.3f}",
            refresh=True
        )

        if global_step % 50 == 0:
            print(f"[Step {global_step:5d}] loss={loss.item():.4f}  edit_acc={edit_acc:.3f}  "
                  f"({edit_count} edits)  lr={current_lr:.2e}")

        loss_log.append({
            "epoch": epoch_num,
            "step": global_step,
            "loss": loss.item(),
            "avg_loss": avg_loss,
            "edit_acc": edit_acc,
            "edit_count": edit_count,
            "lr": current_lr
        })

        if global_step % save_every_steps == 0:
            ckpt_path = checkpoint_dir / f"finetune_step_{global_step}.pt"
            torch.save({
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'step': global_step,
                'epoch': epoch_num,
                'scaler': scaler.state_dict(),
            }, ckpt_path)
            print(f"→ Checkpoint saved: {ckpt_path}")

    # End of epoch
    avg_loss = total_loss / steps_this_epoch if steps_this_epoch > 0 else 0.0
    print(f"Epoch {epoch_num} finished | Avg loss: {avg_loss:.4f} | Useful batches: {useful_batches}")

    # Save epoch checkpoint
    ckpt_path = checkpoint_dir / f"finetune_epoch_{epoch_num}.pt"
    torch.save({
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scaler': scaler.state_dict(),
        'step': global_step,
        'epoch': epoch_num,
    }, ckpt_path)

print("\nTraining finished! Now test full mutated generation.")


=== Finetune Epoch 1 (Mask-Only-Edits Loss) ===


Epoch 1 loss:0.5518 avg:2.2905 edit_acc:0.850:  18%|█▊        | 50/283 [00:32<04:20,  1.12s/it]

[Step    50] loss=0.5518  edit_acc=0.850  (20 edits)  lr=5.00e-06


Epoch 1 loss:0.3349 avg:1.3619 edit_acc:1.000:  36%|███▌      | 101/283 [00:57<01:26,  2.11it/s]

[Step   100] loss=0.2066  edit_acc=1.000  (21 edits)  lr=1.00e-05


...

[Step   750] loss=0.0028  edit_acc=1.000  (17 edits)  lr=4.48e-05


Epoch 3 loss:0.0024 avg:0.0526 edit_acc:1.000:  83%|████████▎ | 235/283 [02:06<00:15,  3.11it/s]

[Step   800] loss=0.4841  edit_acc=0.923  (13 edits)  lr=4.27e-05


Epoch 3 loss:0.0015 avg:0.0513 edit_acc:1.000: 100%|██████████| 283/283 [02:31<00:00,  1.87it/s]


Epoch 3 finished | Avg loss: 0.0513 | Useful batches: 283

Training finished! Now test full mutated generation.
